In [1]:
import oceanbench

oceanbench.__version__

'0.5.1'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# glowcascade_final over the full official start set, cut to NINE lead days.
#
# 52 Wednesday challenger folders of 2024, 20240103 through 20241225, each
# initialised from the as-issued GLO12 nowcast of the Tuesday before it and
# forced by the IFS forecast issued that same Tuesday.
#
# WHY NINE. The IFS forecast package carries lead_day_index 0..9, that is the
# forcing of forecast days 1..10 minus its last day, so the tenth forecast day
# is driven by PERSISTED lead 9 forcing rather than by a forecast. Julien's
# decision of 2026-09-10: the entry scores the nine days that are forced by a
# real IFS forecast and stops there. Nothing is recomputed and no forecast
# zarr is touched: the store still holds ten days per start, this module drops
# the last time step on the way in.
#
# This copy is scored under oceanbench origin/main 7e5ec87 (pending 0.6.0).
import datetime
import pathlib

import xarray

_ROOT = pathlib.Path("/mnt/data/glonet2/ifs21/forecasts/glowcascade_v4_nofilter")
_PATHS = sorted(_ROOT.glob("2024*.zarr"))
_FIRST_DAYS = [datetime.datetime.strptime(p.stem, "%Y%m%d") for p in _PATHS]
_LEAD_DAYS = 9


def _prepared(dataset: xarray.Dataset) -> xarray.Dataset:
    dataset = dataset.isel(time=slice(0, _LEAD_DAYS))
    lead_count = dataset.sizes["time"]
    return dataset.rename({"time": "lead_day_index"}).assign_coords({"lead_day_index": range(lead_count)})


challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [str(p) for p in _PATHS],
    engine="zarr",
    preprocess=_prepared,
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=False,
).assign_coords({"first_day_datetime": _FIRST_DAYS})


### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.066685,0.067074,0.067105,0.067191,0.067572,0.068217,0.069164,0.070447,0.071486
Temperature (°C) [sea_water_potential_temperature]{surface},0.516954,0.517572,0.518081,0.520401,0.524943,0.532741,0.543829,0.556207,0.566318
Salinity (PSU) [sea_water_salinity]{surface},0.579296,0.575312,0.571345,0.567893,0.564353,0.561378,0.558979,0.556789,0.553980
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.118495,0.119159,0.120003,0.121243,0.122872,0.125191,0.128121,0.131443,0.134118
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119529,0.120209,0.121157,0.122546,0.124644,0.127327,0.130479,0.133805,0.136436
Temperature (°C) [sea_water_potential_temperature]{50m},0.864328,0.864212,0.863145,0.862105,0.862715,0.864676,0.868199,0.873047,0.875538
Salinity (PSU) [sea_water_salinity]{50m},0.242673,0.242556,0.242242,0.241924,0.241776,0.241819,0.242015,0.242260,0.242206
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.112322,0.112877,0.113266,0.113785,0.114561,0.115760,0.117399,0.119310,0.120594
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.113282,0.113463,0.113689,0.114110,0.114764,0.115792,0.117345,0.119121,0.120310
Temperature (°C) [sea_water_potential_temperature]{100m},1.058987,1.059697,1.059540,1.059862,1.061136,1.064267,1.069558,1.074729,1.076939


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},41.771338,42.116667,42.403069,42.652253,42.955143,43.296612,43.69088,44.093434,44.299222


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.116794,0.117390,0.117549,0.118062,0.118924,0.119866,0.121895,0.123815,0.125116
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.122688,0.123132,0.123489,0.123367,0.124146,0.124811,0.127055,0.129392,0.131152


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Observations
Temperature (°C) [sea_water_potential_temperature]{surface},0.781183,0.809607,0.779088,0.795881,0.823642,0.865091,0.856932,0.873226,0.901678,153176
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.733545,0.746307,0.766218,0.787539,0.787711,0.802927,0.819161,0.813259,0.808890,82738
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.864431,0.877980,0.861537,0.893271,0.884758,0.889516,0.910534,0.937531,0.938790,1375564
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.779443,0.802420,0.792601,0.794797,0.814043,0.811149,0.835622,0.832940,0.864880,2169693
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.519287,0.533876,0.529089,0.530054,0.542757,0.558050,0.558507,0.568086,0.587058,2702010
Salinity (PSU) [sea_water_salinity]{0-5m},0.255400,0.277587,0.269142,0.304790,0.274514,0.281070,0.270357,0.268357,0.292461,70219
Salinity (PSU) [sea_water_salinity]{5-100m},0.271831,0.268152,0.278186,0.261966,0.301931,0.287505,0.274870,0.288494,0.284055,1173809
Salinity (PSU) [sea_water_salinity]{100-300m},0.126733,0.128648,0.129430,0.129276,0.133114,0.132264,0.137148,0.133497,0.136587,1849747
Salinity (PSU) [sea_water_salinity]{300-600m},0.080449,0.081101,0.080110,0.080965,0.082007,0.083353,0.083770,0.086507,0.087904,2293974
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.048486,0.049453,0.050242,0.051668,0.052918,0.054708,0.056338,0.058149,0.060345,15668173


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},9.968333,19.270504,28.182957,36.806713,45.217258,53.450291,61.531658


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.013604,0.016525,0.018999,0.022179,0.026049,0.030109,0.034416,0.038787,0.041943
Temperature (°C) [sea_water_potential_temperature]{surface},0.171969,0.198631,0.226557,0.255804,0.286950,0.319759,0.355231,0.389838,0.416078
Salinity (PSU) [sea_water_salinity]{surface},0.127489,0.146355,0.162066,0.176213,0.189489,0.204582,0.219817,0.233893,0.245283
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.048676,0.054970,0.061623,0.069179,0.077344,0.085730,0.094673,0.103120,0.109441
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.050302,0.057039,0.063965,0.071421,0.079740,0.088463,0.097286,0.105806,0.112240
Temperature (°C) [sea_water_potential_temperature]{50m},0.303733,0.329035,0.354711,0.383168,0.415377,0.451637,0.490276,0.525848,0.548091
Salinity (PSU) [sea_water_salinity]{50m},0.062892,0.067534,0.072860,0.078779,0.085416,0.092669,0.100268,0.107406,0.112259
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.043706,0.047890,0.052845,0.058361,0.064709,0.071623,0.079082,0.086156,0.091114
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.045748,0.049874,0.054686,0.060171,0.066309,0.073041,0.080227,0.087257,0.092349
Temperature (°C) [sea_water_potential_temperature]{100m},0.272786,0.304714,0.338567,0.377323,0.419525,0.464862,0.512242,0.555533,0.582722


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},31.937032,32.901421,33.672485,34.472153,35.302007,36.173959,37.070637,37.852782,38.495524


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.043767,0.050630,0.057699,0.065028,0.072229,0.079699,0.087330,0.094394,0.099270
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.045601,0.053419,0.061595,0.069579,0.077308,0.085394,0.093342,0.100495,0.105251


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},3.995353,7.912031,12.019297,16.482376,21.368671,26.69285,32.46653
